# DML による CATE の推定と感度分析

- **作成者**: -
- **作成日**: 2026-11-15
- **目的**: 処置 `T`（連続値）が結果 `Y` に与える効果を、交絡因子 `X1`〜`X15` ごとの異質性（CATE）まで含めて
  econml の Double Machine Learning（DML）で推定し、未観測交絡に対する頑健さを感度分析で確かめる。

## 前提（DAG の作成と estimand の同定は完了済み）

| 変数 | 役割 | 推定での扱い |
|---|---|---|
| `Y` | 結果変数 | |
| `T` | 処置変数（連続値） | 効果 = T を1単位増やしたときの Y の変化 |
| `X1`〜`X15` | 交絡因子（調整する）かつ効果修飾因子 | econml の `X`（nuisance と CATE の両方に使う） |
| `W1`〜`W30` | 結果の予測精度を上げる共変量（交絡因子ではない） | econml の `W`（nuisance にだけ使う） |

- **estimand**: X（と W）による backdoor 調整で識別される CATE θ(x) = ∂E[Y | do(T=t), X=x] / ∂t。
- **識別の仮定**: 条件付き無交絡（X を調整すれば未観測交絡は無い）、正値性（どの X でも T がばらつく）、
  SUTVA（個体間の干渉なし）。無交絡は検証できないので、後半の感度分析で「仮定が崩れたらどうなるか」を見る。
- **モデルの仮定**: 部分線形モデル Y = θ(X)·T + g(X, W) + ε（効果は T について線形）。

## 流れ

1. データの読み込みと inf・欠損の処理（このノートブックのセル内で行う）
2. ベースライン（OLS）の推定
3. nuisance モデル（E[Y|X,W]・E[T|X,W]）の Optuna によるチューニング
4. DML（`LinearDML`・`CausalForestDML`）による推定。nuisance は **cross-fitting**（K-fold、または時系列なら時間順のブロック）で学習する
5. 推定結果の可視化（合成データなら真値との比較も）
6. 感度分析: E-value／econml の omitted variable bias 感度分析／DoWhy の反証テスト
7. まとめ

図はノートブックに表示し、同じものを `outputs/figures/dml/<RUN_ID>/` に、表を `outputs/tables/dml/<RUN_ID>/` に保存する。

In [ ]:
from __future__ import annotations

import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.exceptions import DataConversionWarning
from sklearn.impute import SimpleImputer

# DoWhy が econml の推定器に1列の2次元配列を渡すたびに出る警告（結果には影響しない）を表示しない
warnings.filterwarnings("ignore", category=DataConversionWarning)

# リポジトリのルート（pyproject.toml がある場所）を探し、src を import できるようにする
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from causal_inference import plots  # noqa: E402
from causal_inference.estimators import (  # noqa: E402
    ESTIMATOR_LABELS,
    build_estimator,
    fit_estimator,
    linear_dml_coefficients,
    nuisance_scores,
    summarize_effects,
    tune_causal_forest,
)
from causal_inference.evaluation import baseline_estimates, compare_with_truth  # noqa: E402
from causal_inference.nuisance import (  # noqa: E402
    DEFAULT_NUISANCE_PARAMS,
    make_cv,
    make_nuisance_model,
    tune_nuisance,
)
from causal_inference.output import OutputSaver  # noqa: E402
from causal_inference.sensitivity import (  # noqa: E402
    default_refuters,
    e_value_table,
    fit_dowhy_wrapper,
    ovb_grid,
    ovb_summary,
    run_refuters,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## パラメータ

papermill で実行するときは `-p 名前 値` で上書きできる（このセルに `parameters` タグを付けている）。
リストは `-y "UNOBSERVED_STRENGTHS: [0.1, 0.3]"` のように YAML で渡す（`-p` だと文字列になる。
文字列で渡されたときも、次のセルで JSON として読み直す）。

In [ ]:
# --- データ ---
DATA_PATH = "data/synthetic/dml/dummy_dml.csv"  # リポジトリのルートからの相対パス
TRUTH_PATH = "data/synthetic/dml/dummy_dml_truth.csv"  # 真の CATE（合成データ用。実データでは ""）
OUTCOME = "Y"
TREATMENT = "T"
X_COLS = [f"X{i}" for i in range(1, 16)]  # 交絡因子かつ効果修飾因子
W_COLS = [f"W{i}" for i in range(1, 31)]  # nuisance にだけ使う共変量
IMPUTE_STRATEGY = "median"  # X・W の欠損の補完方法（"median" / "mean"）

TIME_COL = ""  # 時刻の列（時系列のとき。指定すると時間順に並べ替える。空なら今の行の順）

# --- cross-fitting と推定 ---
CV_STRATEGY = "kfold"  # "kfold"（行が独立）/ "timeseries"（時間順の連続ブロック）
N_SPLITS = 5  # cross-fitting の分割数（2以上）
CV_GAP = 0  # timeseries のとき、検証ブロックの前後で学習から除く行数（自己相関の長さが目安）
MC_ITERS = 1  # 分割を変えて cross-fitting を繰り返す回数（timeseries では1のみ）
SEED = 0
ALPHA = 0.05  # CI の有意水準（95% CI）
ESTIMATORS = ["linear_dml", "causal_forest_dml"]
FOREST_N_ESTIMATORS = 500  # CausalForestDML の木の数
TUNE_FOREST = True  # CausalForestDML の最終段を econml の tune（R-score）で選ぶ
NUISANCE_N_JOBS = 4  # LightGBM のスレッド数（全スレッドにするとスレッドの取り合いで遅くなる）

# --- Optuna ---
USE_OPTUNA = True  # False なら nuisance は既定パラメータ
N_TRIALS = 30  # nuisance モデル1つあたりの試行回数
TUNING_TIMEOUT = None  # nuisance モデル1つあたりの探索の上限時間（秒）

# --- 感度分析 ---
SENSITIVITY_C = 0.05  # econml の感度分析の要約に使う交絡の強さ（c_y と c_t に同じ値を使う）
OVB_C_MAX = 0.30  # 等高線図の c_y・c_t の上限（robustness value がこれより大きければ広げる）
RUN_REFUTERS = True  # DoWhy の反証テスト（推定し直すので時間がかかる）
REFUTER_ESTIMATOR = "linear_dml"
NUM_SIMULATIONS = 10  # 反証テスト1つあたりの推定し直しの回数
REFUTER_MAX_ROWS = 3000  # 反証テストに使う行数の上限（None なら全行）
REFUTER_N_JOBS = 1  # 反証テストの並列数
UNOBSERVED_STRENGTHS = [0.05, 0.1, 0.2]  # 模擬的な未観測交絡の強さ

# --- 出力 ---
RUN_ID = ""  # 出力フォルダ名（空なら実行日時）

In [ ]:
# papermill の -p で渡されたリストは文字列になるので、JSON として読み直す
X_COLS, W_COLS, ESTIMATORS, UNOBSERVED_STRENGTHS = (
    json.loads(value) if isinstance(value, str) else list(value)
    for value in (X_COLS, W_COLS, ESTIMATORS, UNOBSERVED_STRENGTHS)
)

plots.setup_style()
saver = OutputSaver.create("dml", RUN_ID or None)
print(f"図の保存先: {saver.figure_dir.relative_to(REPO_ROOT)}")
print(f"表の保存先: {saver.table_dir.relative_to(REPO_ROOT)}")


def show_and_save(fig: plt.Figure, filename: str) -> None:
    # ノートブックに表示してから保存し、図を閉じる（同じ図が二重に表示されないように）
    display(fig)
    saver.figure(fig, filename)


def show_table(frame: pd.DataFrame, filename: str, **kwargs) -> None:
    display(frame)
    saver.table(frame, filename, **kwargs)

## 1. データの読み込みと inf・欠損の処理

どの変数にも欠損（NaN）や ±inf が含まれうる前提で、次のように処理する。

| 対象 | 処理 | 理由 |
|---|---|---|
| 数値でない値 | NaN にする | 推定は数値だけを扱う |
| ±inf | NaN にする | 推定・補完ができない |
| `Y`・`T` が NaN の行 | 除外する | 結果・処置が分からない行は推定に使えない |
| すべて NaN の `X`・`W` の列 | 除外して警告する | 補完できない |
| `X`・`W` の NaN | 中央値で補完する | econml の最終段・DoWhy 連携は X の欠損を受け付けない |

欠損指示変数は加えない（X に加えると効果修飾の変数が変わり、CATE の意味が変わるため）。
補完した値は真の値と違うので、欠損が多いと交絡の調整が弱くなりうる（まとめの注意点に書く）。

In [ ]:
raw = pd.read_csv(REPO_ROOT / DATA_PATH)
truth_raw = pd.read_csv(REPO_ROOT / TRUTH_PATH) if TRUTH_PATH else None

analysis_cols = [OUTCOME, TREATMENT, *X_COLS, *W_COLS]
absent = [c for c in analysis_cols if c not in raw.columns]
if absent:
    raise KeyError(f"データに無い列があります: {absent}")
if truth_raw is not None and len(truth_raw) != len(raw):
    raise ValueError(f"真値の行数 {len(truth_raw)} がデータの行数 {len(raw)} と違います")

# 時系列のときは時間順に並べ替える（時刻が欠損の行は位置が決まらないので除外する）
time_index = None
if TIME_COL:
    if TIME_COL not in raw.columns:
        raise KeyError(f"時刻の列がありません: {TIME_COL}")
    time_values = raw[TIME_COL]
    if not pd.api.types.is_numeric_dtype(time_values):
        time_values = pd.to_datetime(time_values, errors="coerce")
    order = time_values.dropna().sort_values(kind="stable").index
    if len(order) < len(raw):
        warnings.warn(f"時刻が欠損の {len(raw) - len(order)} 行を除外します", stacklevel=1)
    raw = raw.loc[order].reset_index(drop=True)
    truth_raw = truth_raw.loc[order].reset_index(drop=True) if truth_raw is not None else None
    time_index = time_values.loc[order].reset_index(drop=True)
    print(f"{TIME_COL} で時間順に並べ替えました: {time_index.iloc[0]} 〜 {time_index.iloc[-1]}")
elif CV_STRATEGY == "timeseries":
    print("TIME_COL が空なので、今の行の順を時間順とみなします")

# 数値に変換する（数値でない値は NaN）
numeric = raw[analysis_cols].apply(pd.to_numeric, errors="coerce")
roles = {
    OUTCOME: "結果",
    TREATMENT: "処置",
    **dict.fromkeys(X_COLS, "X"),
    **dict.fromkeys(W_COLS, "W"),
}
quality = pd.DataFrame(
    {
        "role": pd.Series(roles),
        "n_nan": numeric.isna().sum(),
        "n_pos_inf": numeric.eq(np.inf).sum(),
        "n_neg_inf": numeric.eq(-np.inf).sum(),
        "n_non_numeric": (numeric.isna() & raw[analysis_cols].notna()).sum(),
    }
).rename_axis("column")
quality["nonfinite_rate"] = (quality["n_nan"] + quality["n_pos_inf"] + quality["n_neg_inf"]) / len(
    raw
)
print(f"読み込んだ行数: {len(raw):,}、列数: {len(analysis_cols)}")
print(f"欠損・inf を含む列: {(quality['nonfinite_rate'] > 0).sum()} / {len(quality)}")
show_table(quality.reset_index(), "data_quality")

In [ ]:
# ±inf を NaN にそろえる
cleaned = numeric.replace([np.inf, -np.inf], np.nan)

# Y・T が NaN の行は除外する
keep = cleaned[[OUTCOME, TREATMENT]].notna().all(axis=1).to_numpy()

# すべて NaN の列は補完できないので除外する
all_nan_cols = [c for c in [*X_COLS, *W_COLS] if cleaned.loc[keep, c].isna().all()]
if all_nan_cols:
    warnings.warn(f"すべて欠損の列を除外します: {all_nan_cols}", stacklevel=1)
x_cols = [c for c in X_COLS if c not in all_nan_cols]
w_cols = [c for c in W_COLS if c not in all_nan_cols]
if not x_cols:
    raise ValueError("使える X の列がありません")

data = cleaned.loc[keep].reset_index(drop=True)
truth = truth_raw.loc[keep].reset_index(drop=True) if truth_raw is not None else None
if time_index is not None:
    time_index = time_index.loc[keep].reset_index(drop=True)

# X・W の欠損を補完する（中央値などは推定に使う行で計算する）
feature_cols = [*x_cols, *w_cols]
n_imputed = data[feature_cols].isna().sum()
imputer = SimpleImputer(strategy=IMPUTE_STRATEGY)
features = pd.DataFrame(imputer.fit_transform(data[feature_cols]), columns=feature_cols)

Y = data[OUTCOME]
T = data[TREATMENT]
X = features[x_cols]
W = features[w_cols] if w_cols else None
if T.std() == 0:
    raise ValueError("T が一定の値なので、効果を推定できません")

print(f"推定に使う行数: {len(data):,}（Y・T の欠損で除外: {(~keep).sum():,} 行）")
print(f"X: {len(x_cols)} 列、W: {len(w_cols)} 列")
print(f"補完した値: {int(n_imputed.sum()):,} 個（{IMPUTE_STRATEGY}）")

In [ ]:
display(pd.concat([Y, T], axis=1).describe().T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
axes[0].hist(Y, bins=50)
axes[0].set_title(f"{OUTCOME} の分布")
axes[1].hist(T, bins=50)
axes[1].set_title(f"{TREATMENT} の分布")
sample = np.random.default_rng(SEED).choice(len(Y), min(len(Y), 3000), replace=False)
axes[2].scatter(T.iloc[sample], Y.iloc[sample], s=5, alpha=0.3)
axes[2].set_xlabel(TREATMENT)
axes[2].set_ylabel(OUTCOME)
axes[2].set_title(f"{TREATMENT} と {OUTCOME}（調整前の関係、{len(sample):,} 点）")
for ax in axes[:2]:
    ax.set_ylabel("件数")
fig.suptitle(f"推定に使うデータ（n = {len(Y):,}）")
show_and_save(fig, "01_data_overview")

## 2. ベースライン（OLS）

- `naive_ols`: Y ~ T（調整なし）。交絡があるとずれる。
- `adjusted_ols`: Y ~ T + X + W（線形の調整）。非線形の交絡・効果の異質性は扱えない。

In [ ]:
baselines = baseline_estimates(Y, T, X, W, alpha=ALPHA)
display(baselines)

## 3. nuisance モデルの Optuna によるチューニング

DML は、結果のモデル E[Y|X,W] と処置のモデル E[T|X,W] の残差どうしを回帰して効果を求める。
nuisance の予測が悪いと残差に交絡が残るので、LightGBM のハイパーパラメータを Optuna で探す。

- 目的関数: cross-fitting と同じ分割（`CV_STRATEGY`・`N_SPLITS`・`CV_GAP`）での out-of-fold MSE
  （fold 平均）
- 探索空間: 既存の LightGBM の探索空間（`modeling.models.gbdt`）
- 注意: 全データの CV でパラメータを選ぶので、推定に使うデータを一度見ている。予測精度だけを見ていて、
  処置効果は直接見ていないので、通常は問題にならない。

In [ ]:
features_xw = pd.concat([X, W], axis=1) if W is not None else X
cv = make_cv(N_SPLITS, SEED, strategy=CV_STRATEGY, gap=CV_GAP)
cv_options = {"n_splits": N_SPLITS, "mc_iters": MC_ITERS, "cv_strategy": CV_STRATEGY, "gap": CV_GAP}
if CV_STRATEGY == "timeseries":
    # 時系列の分割は推定（econml）にもこのまま渡すので、ここで分割の形を確認しておく
    show_and_save(plots.plot_cv_folds(cv, len(Y)), "00_cv_folds")
base_params = {**DEFAULT_NUISANCE_PARAMS, "n_jobs": NUISANCE_N_JOBS}

tuning_results = {}
if USE_OPTUNA:
    for name, target in ((OUTCOME, Y), (TREATMENT, T)):
        start = time.perf_counter()
        tuning_results[name] = tune_nuisance(
            features_xw,
            target,
            cv=cv,
            n_trials=N_TRIALS,
            seed=SEED,
            target_name=name,
            timeout=TUNING_TIMEOUT,
            base_params=base_params,
        )
        result = tuning_results[name]
        note = "。既定より良い試行が無いので既定を使う" if result.used_baseline else ""
        print(
            f"{name}: MSE {result.baseline_value:.4f}（既定）→ {result.best_value:.4f}（採用）、"
            f"{time.perf_counter() - start:.0f} 秒{note}"
        )
    params_y = tuning_results[OUTCOME].best_params
    params_t = tuning_results[TREATMENT].best_params
    show_and_save(plots.plot_tuning_history(list(tuning_results.values())), "02_tuning_history")
    saver.table(pd.concat([r.trials for r in tuning_results.values()]), "tuning_trials")
else:
    params_y = params_t = dict(base_params)

nuisance_params = pd.DataFrame(
    {f"model_y（{OUTCOME}）": params_y, f"model_t（{TREATMENT}）": params_t}
)
show_table(nuisance_params.rename_axis("param").reset_index(), "nuisance_params")

## 4. DML による推定（cross-fitting あり）

- `LinearDML`: θ(X) を X の線形関数で表す。係数と CI を解析的に出せる。
- `CausalForestDML`: θ(X) を因果フォレストで非線形に推定する（`TUNE_FOREST` なら最終段を R-score で選ぶ）。

どちらも nuisance を `N_SPLITS` 分割の cross-fitting で学習し、各行の残差はその行を学習に
使っていないモデルで計算する。分割の方法は `CV_STRATEGY` で選ぶ。

| `CV_STRATEGY` | 分割 | 使う場面 |
|---|---|---|
| `kfold` | 行をシャッフルした K-fold（`MC_ITERS` 回、分割を変えて繰り返せる） | 行が互いに独立 |
| `timeseries` | 時間順の連続ブロック。検証ブロックの前後 `CV_GAP` 行を学習から除く | 時系列（自己相関がある） |

`timeseries` では、学習に検証ブロックより**未来**の行も使う（DML の cross-fitting は全行の
out-of-fold の残差が必要で、過去だけで学習する分割では最初のブロックの残差が作れないため）。
nuisance の推定に未来の情報を使うことは、予測ではなく効果の推定なので問題にならないが、
自己相関で検証ブロックの情報が学習に漏れないよう、`CV_GAP` に自己相関が続く長さ（行数）を指定する。

In [ ]:
estimators = {}
summaries = {}
for name in ESTIMATORS:
    start = time.perf_counter()
    extra = {"n_estimators": FOREST_N_ESTIMATORS} if name == "causal_forest_dml" else {}
    estimator = build_estimator(
        name,
        make_nuisance_model(params_y, SEED),
        make_nuisance_model(params_t, SEED),
        seed=SEED,
        **cv_options,
        **extra,
    )
    if name == "causal_forest_dml" and TUNE_FOREST:
        chosen = tune_causal_forest(estimator, Y, T, X, W)
        print(f"{ESTIMATOR_LABELS[name]} の最終段のパラメータ: {chosen}")
    fit_estimator(estimator, Y, T, X, W)
    estimators[name] = estimator
    summaries[name] = summarize_effects(ESTIMATOR_LABELS[name], estimator, X, alpha=ALPHA)
    print(
        f"{ESTIMATOR_LABELS[name]}: ATE {summaries[name].ate:.4f} "
        f"[{summaries[name].ate_lower:.4f}, {summaries[name].ate_upper:.4f}]、"
        f"{time.perf_counter() - start:.0f} 秒"
    )

In [ ]:
# nuisance モデルの out-of-fold の決定係数（fold ごと）
scores = pd.concat(
    [
        nuisance_scores(est).assign(estimator=ESTIMATOR_LABELS[name])
        for name, est in estimators.items()
    ]
)
display(scores.groupby(["estimator", "model"])["r2"].agg(["mean", "min", "max"]))
saver.table(scores, "nuisance_scores")

## 5. 推定結果

### 5.1 ATE

In [ ]:
dml_ate = pd.DataFrame([s.to_row() for s in summaries.values()])
ate_table = pd.concat([baselines, dml_ate], ignore_index=True)
show_table(ate_table, "ate")

true_ate = float(truth["true_cate"].mean()) if truth is not None else None
show_and_save(plots.plot_ate_comparison(ate_table, true_ate=true_ate), "03_ate_comparison")

### 5.2 CATE（効果の異質性）

In [ ]:
cate_frames = {s.name: s.cate for s in summaries.values()}
true_cate = truth["true_cate"].to_numpy() if truth is not None else None

show_and_save(plots.plot_cate_distribution(cate_frames, true_cate), "04_cate_distribution")
show_and_save(plots.plot_cate_by_feature(X, cate_frames, true_cate), "05_cate_by_feature")

# 行ごとの CATE（個体ごとの値なので、実データでは外部に出さない）
cate_wide = pd.concat([frame.add_prefix(f"{name}_") for name, frame in cate_frames.items()], axis=1)
saver.table(cate_wide, "cate_by_row")

In [ ]:
if "linear_dml" in estimators:
    coefficients = linear_dml_coefficients(estimators["linear_dml"], x_cols, alpha=ALPHA)
    show_table(coefficients, "linear_dml_coefficients")
    show_and_save(
        plots.plot_linear_coefficients(coefficients, "LinearDML の θ(X) の係数"),
        "06_linear_dml_coefficients",
    )

if "causal_forest_dml" in estimators:
    importances = pd.Series(estimators["causal_forest_dml"].feature_importances_, index=x_cols)
    saver.table(
        importances.rename("importance").rename_axis("feature").reset_index(),
        "causal_forest_importance",
    )
    show_and_save(
        plots.plot_feature_importance(importances, "CausalForestDML の変数の重要度"),
        "07_causal_forest_importance",
    )

### 5.3 真の CATE との比較（合成データのみ）

真値が分かる合成データでだけ使える評価。実データ（`TRUTH_PATH = ""`）では飛ばす。

In [ ]:
if truth is not None:
    truth_metrics = pd.DataFrame([compare_with_truth(s, true_cate) for s in summaries.values()])
    show_table(truth_metrics, "truth_comparison")
    show_and_save(
        plots.plot_cate_true_vs_estimated(true_cate, cate_frames, seed=SEED),
        "08_cate_true_vs_estimated",
    )
else:
    truth_metrics = None
    print("真値が無いので比較しません")

## 6. 感度分析

### 6.1 E-value

推定した効果を「無かったこと」にするために、未観測の交絡因子が処置と結果の**両方**に持つ必要がある関連の
強さ（リスク比のスケール）の最小値。大きいほど、未観測交絡で説明しきるのが難しい。

- Y が連続なので、効果を SD(Y) で標準化した d から RR ≈ exp(0.91·d) と近似する（VanderWeele, 2017）。
- T が連続なので、T を「1単位」と「1標準偏差」動かしたときの効果のそれぞれで計算する。
- 「CI の限界」は CI の 0 に近い側の E-value（CI が 0 を含めば 1）。

In [ ]:
sd_y, sd_t = float(Y.std()), float(T.std())
deltas = {"T +1単位": 1.0, f"T +1SD（{sd_t:.2f}）": sd_t}
e_values = e_value_table(dml_ate, sd_outcome=sd_y, deltas=deltas)
show_table(e_values, "e_values")
show_and_save(plots.plot_e_values(e_values), "09_e_values")

first = e_values[e_values["delta_label"] == list(deltas)[1]].iloc[0]
show_and_save(
    plots.plot_e_value_curve(
        first["rr"], first["e_value_point"], f"{first['estimator']}、{first['delta_label']}"
    ),
    "10_e_value_curve",
)

### 6.2 omitted variable bias の感度分析（econml、Chernozhukov et al. 2022）

未観測の交絡因子が、結果の残差の分散を **c_y**、処置の残差（Riesz representer）の分散を **c_t** の割合だけ
説明するとしたときの ATE の範囲を求める。

- `theta_lower`〜`theta_upper`: 交絡による偏りだけを考えた点推定の範囲
- `ci_lower`〜`ci_upper`: さらに標本誤差も考えた範囲
- **robustness value**（`rv_ci`）: c_y = c_t のとき、CI が 0 を含むようになる交絡の強さ。大きいほど頑健。

ここでの ATE（`theta`）は部分線形モデルの残差回帰の推定値で、5.1 の CATE の平均とは少し違うことがある。

In [ ]:
ovb_rows = []
for name, estimator in estimators.items():
    ovb_rows.append(
        {
            "estimator": ESTIMATOR_LABELS[name],
            **ovb_summary(estimator, c_y=SENSITIVITY_C, c_t=SENSITIVITY_C, alpha=ALPHA),
        }
    )
ovb_table = pd.DataFrame(ovb_rows)
show_table(ovb_table, "ovb_sensitivity")

for name, estimator in estimators.items():
    row = ovb_table[ovb_table["estimator"] == ESTIMATOR_LABELS[name]].iloc[0]
    # 効果が 0 になる境界と robustness value が図に入るよう、格子の上限を広げる（1 未満）
    c_max = min(0.95, max(OVB_C_MAX, 1.3 * row["rv_ci"]))
    grid = ovb_grid(estimator, c_values=np.linspace(0.0, c_max, 21), alpha=ALPHA)
    saver.table(grid, f"ovb_grid_{name}")
    fig = plots.plot_ovb_contour(grid, theta=row["theta"], robustness_value=row["rv_ci"])
    fig.axes[0].set_title(f"{ESTIMATOR_LABELS[name]}: " + fig.axes[0].get_title())
    show_and_save(fig, f"11_ovb_contour_{name}")

### 6.3 DoWhy の反証テスト

推定の前提を意図的に崩したデータで推定し直し、推定値が期待どおりに振る舞うかを見る。

| テスト | 期待される結果 |
|---|---|
| 偽の処置（T を並べ替え） | 効果が 0 に近くなる |
| 無関係な共変量を追加 | 効果がほとんど変わらない |
| 部分標本 | 効果がほとんど変わらない |
| 偽の結果（Y を無関係なノイズに） | 効果が 0 に近くなる |
| 模擬的な未観測交絡 | 期待値は無い。どれだけ動くかを見る |

`judgement` は目安（元の推定値の 10% 以内なら OK）。`p_value` はシミュレーション回数が少ないと不安定。
推定し直しを繰り返すので、行数を `REFUTER_MAX_ROWS` までに減らして実行する（元の推定値もその行で計算し直す）。

In [ ]:
if RUN_REFUTERS:
    rows = np.arange(len(Y))
    if REFUTER_MAX_ROWS is not None and len(rows) > REFUTER_MAX_ROWS:
        if CV_STRATEGY == "timeseries":
            # 時系列は時間の連続性を保つため、最後の連続した期間を使う
            rows = rows[-REFUTER_MAX_ROWS:]
        else:
            rng = np.random.default_rng(SEED)
            rows = np.sort(rng.choice(rows, REFUTER_MAX_ROWS, replace=False))
    extra = (
        {"n_estimators": FOREST_N_ESTIMATORS} if REFUTER_ESTIMATOR == "causal_forest_dml" else {}
    )
    refute_base = build_estimator(
        REFUTER_ESTIMATOR,
        make_nuisance_model(params_y, SEED),
        make_nuisance_model(params_t, SEED),
        seed=SEED,
        **cv_options,
        **extra,
    )
    start = time.perf_counter()
    wrapper = fit_dowhy_wrapper(
        refute_base,
        Y.iloc[rows],
        T.iloc[rows],
        X.iloc[rows],
        None if W is None else W.iloc[rows],
        outcome_name=OUTCOME,
        treatment_name=TREATMENT,
    )
    refuters = default_refuters(
        num_simulations=NUM_SIMULATIONS,
        seed=SEED,
        n_jobs=REFUTER_N_JOBS,
        unobserved_strengths=UNOBSERVED_STRENGTHS,
    )
    refutations = run_refuters(wrapper, refuters)
    print(
        f"{ESTIMATOR_LABELS[REFUTER_ESTIMATOR]}、{len(rows):,} 行、"
        f"{time.perf_counter() - start:.0f} 秒"
    )
    show_table(refutations, "refutations")
    show_and_save(plots.plot_refutations(refutations), "12_refutations")
else:
    refutations = None
    print("RUN_REFUTERS = False なので、反証テストは実行しません")

## 7. まとめ

下のセルは、この実行の数値から要約を作る。解釈の目安:

- **ATE**: DML の CI が 0 を含まなければ、仮定の下で効果があると言える。ベースライン（OLS）との差は、
  非線形の交絡・効果の異質性をどれだけ扱えたかの目安。
- **CATE**: `LinearDML` の係数の CI が 0 を含まない変数、`CausalForestDML` の重要度が高い変数で効果が変わる。
- **感度分析**: E-value・robustness value が小さいと、弱い未観測交絡でも結論が変わりうる。反証テストで
  「要確認」があれば、モデルの設定（nuisance の精度・cross-fitting の分割数）やデータを見直す。

注意点:

- 結論は「X・W を調整すれば未観測交絡は無い」という識別の仮定に依存する。感度分析はこの仮定が崩れたときの
  影響の大きさを示すだけで、仮定が正しいことは示さない。
- 部分線形モデル（効果は T について線形）を仮定している。T の範囲によって効果が変わる場合は扱えない。
- X・W の欠損は中央値で補完しているので、欠損が多い変数は交絡の調整が弱くなりうる。
- 行ごとの CATE（`cate_by_row.csv`）は個体ごとの値なので、実データでは外部に出さない。
- 時系列（`CV_STRATEGY = "timeseries"`）では、econml の CI は残差が互いに独立だと仮定して計算するので、
  残差に自己相関が残ると CI が狭すぎる（過信する）ことがある。DoWhy の反証テストのうち、
  処置の並べ替え・部分標本は時間の並びを崩すので、目安として見る。

In [ ]:
def ci_text(lower: float, upper: float) -> str:
    return f"[{lower:.3f}, {upper:.3f}]"


cv_text = f"{CV_STRATEGY} {N_SPLITS} 分割" + (
    f"・gap {CV_GAP}" if CV_STRATEGY == "timeseries" else ""
)
lines = [f"### 結果の要約（n = {len(Y):,}、cross-fitting: {cv_text} × {MC_ITERS} 回）", ""]
lines.append(
    f"| 推定器 | ATE | {1 - ALPHA:.0%} CI | E-value（+1SD、点 / CI） | robustness value（CI） |"
)
lines.append("|---|---|---|---|---|")
sd_label = list(deltas)[1]
for summary in summaries.values():
    is_row = (e_values["estimator"] == summary.name) & (e_values["delta_label"] == sd_label)
    ev = e_values[is_row].iloc[0]
    rv = ovb_table.loc[ovb_table["estimator"] == summary.name, "rv_ci"].iloc[0]
    lines.append(
        f"| {summary.name} | {summary.ate:.3f} | {ci_text(summary.ate_lower, summary.ate_upper)} "
        f"| {ev['e_value_point']:.2f} / {ev['e_value_ci']:.2f} | {rv:.3f} |"
    )
for row in baselines.itertuples(index=False):
    ci = ci_text(row.ate_lower, row.ate_upper)
    lines.append(f"| {row.estimator}（ベースライン） | {row.ate:.3f} | {ci} | - | - |")
if truth_metrics is not None:
    lines += ["", f"真の ATE（標本平均）: {true_ate:.3f}", ""]
    for row in truth_metrics.itertuples(index=False):
        covered = "含む" if row.ate_covered else "含まない"
        lines.append(
            f"- {row.estimator}: ATE の誤差 {row.ate_error:+.3f}（CI が真値を{covered}）、"
            f"CATE の相関 {row.cate_corr:.3f}・RMSE {row.cate_rmse:.3f}・"
            f"CI の被覆率 {row.cate_coverage:.1%}"
        )
if refutations is not None:
    flagged = refutations[refutations["judgement"] == "要確認"]["refuter"].tolist()
    failed = refutations[refutations["error"] != ""]["refuter"].tolist()
    refute_text = f"反証テスト: 要確認 {len(flagged)} 件 {flagged or ''}"
    lines += ["", f"{refute_text}、失敗 {len(failed)} 件 {failed or ''}"]
display(Markdown("\n".join(lines)))

In [ ]:
print(f"保存したファイル（{len(saver.paths)} 個）:")
for path in saver.paths:
    print(" -", path.relative_to(REPO_ROOT))